# einops.reduce — procedural drill

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `einops-reduce`. When a test cell passes, your progress is reported back to your account.

**What you'll practice.** Five `einops.reduce` patterns that ramp from single-axis mean → multi-axis global pool → keepdim broadcast → decomposed average pool → softmax stabilization. Read the docstring, fill the function body, run the test cell. The solution sits in the collapsed `<details>` block below each exercise.

**Per-exercise structure** (Doughty et al. ACE 2024 — `[Bloom level] + [LO] + [Keywords] + [KCs]`):
Each exercise begins with a yaml block stating its Bloom cognitive level, learning objective, keywords, and the knowledge components (KCs) it targets. This makes the cognitive demand explicit instead of buried.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import torch.nn.functional as F
import einops
from einops import reduce

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Einops: Reduce` subtopic.
You can copy the token from your Delta Drills account page.

This drill exercises the **atom `einops-reduce`**, which bridges to the bank subtopic `Einops: Reduce` for EWMA state. Completing all 5 exercises triggers a single `arena-rating` beacon at the end of the notebook.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "einops-reduce"
DD_SUBTOPIC = "Einops: Reduce"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

# Track which exercises passed in this session.
_dd_passed = set()

## einops.reduce — quick refresher

`reduce(tensor, pattern, reduction, **axes_lengths)` collapses named axes:
1. **Single-axis drop** — `'h w c -> h w'` with `'mean'` averages channels.
2. **Multi-axis drop** — `'b c h w -> b'` reduces three axes at once.
3. **Keepdim placeholder** — `'b c h w -> b c () ()'` keeps size-1 axes for broadcasting.
4. **Decompose-then-reduce** — `'b c (h h2) (w w2) -> b c h w'` with `h2=2, w2=2` does 2×2 pooling.

Reduction strings: `'mean'`, `'sum'`, `'max'`, `'min'`, `'prod'`, `'any'`, `'all'`, or a callable.
Any axis that appears on the left but not on the right is reduced over.

### Exercise 1 — channel mean

> ```yaml
> Difficulty: ⚪⚪⚪⚪⚪
> Bloom level: Remember
> LO: Recall the `reduce(x, pattern, op)` call shape for collapsing a single named axis.
> Keywords: aggregator, single-axis-drop, mean
> ```

**KCs targeted:** `reduce-pick-aggregator`

Implement `ex1_channel_mean(x)` to average a `(h, w, c)` image over its channel axis. Output shape: `(h, w)`.

Use `einops.reduce(...)` with the `'mean'` reduction — not `x.mean(dim=-1)`. The point is to write the pattern.

In [ ]:
def ex1_channel_mean(x: Tensor) -> Tensor:
    """Reduce `x` of shape (h, w, c) to (h, w) by averaging channels."""
    raise NotImplementedError()


def _test_ex1():
    x = t.arange(4 * 5 * 3).reshape(4, 5, 3).float()
    y = ex1_channel_mean(x)
    assert y.shape == (4, 5), f'expected (4,5), got {y.shape}'
    assert t.allclose(y, x.mean(dim=-1)), 'values differ from x.mean(dim=-1)'
    _dd_passed.add('ex1')
    print("ex1 ✓")

_test_ex1()

<details><summary>Solution</summary>

```python
def ex1_channel_mean(x: Tensor) -> Tensor:
    return reduce(x, 'h w c -> h w', 'mean')
```
</details>

### Exercise 2 — per-image global mean (multi-axis drop)

> ```yaml
> Difficulty: 🔴⚪⚪⚪⚪
> Bloom level: Apply
> LO: Apply the reduce pattern to collapse three axes in a single call (global-mean per batch item).
> Keywords: multi-axis-drop, global-pool, mean
> ```

**KCs targeted:** `reduce-multi-axis`

Implement `ex2_global_mean(x)` to compute one scalar per image: the mean of all channel × spatial values.

Input shape: `(b, c, h, w)`. Output shape: `(b,)`.

Any axis name that appears on the left but **not** on the right is reduced over — so you drop `c`, `h`, and `w` at once.

In [ ]:
def ex2_global_mean(x: Tensor) -> Tensor:
    """Reduce (b, c, h, w) → (b,) by averaging over c, h, w."""
    raise NotImplementedError()


def _test_ex2():
    x = t.arange(2 * 3 * 4 * 5).reshape(2, 3, 4, 5).float()
    y = ex2_global_mean(x)
    assert y.shape == (2,), f'expected (2,), got {y.shape}'
    assert t.allclose(y, x.mean(dim=(1, 2, 3))), 'values differ from x.mean(dim=(1,2,3))'
    _dd_passed.add('ex2')
    print("ex2 ✓")

_test_ex2()

<details><summary>Solution</summary>

```python
def ex2_global_mean(x: Tensor) -> Tensor:
    return reduce(x, 'b c h w -> b', 'mean')
```
</details>

### Exercise 3 — per-image spatial max (keepdim with ())

> ```yaml
> Difficulty: 🔴🔴⚪⚪⚪
> Bloom level: Apply
> LO: Apply `()` on the output side to preserve a collapsed axis as size-1 for downstream broadcasting.
> Keywords: keepdim, placeholder-axis, max
> ```

**KCs targeted:** `reduce-keepdim-broadcast`

Implement `ex3_spatial_max(x)` to compute the per-(batch, channel) spatial maximum, keeping the H and W axes as size-1 placeholders so the result broadcasts back against `x`.

Input shape: `(b, c, h, w)`. Output shape: `(b, c, 1, 1)`.

Use `()` on the right side of the pattern wherever you want a size-1 axis to remain instead of being dropped.

In [ ]:
def ex3_spatial_max(x: Tensor) -> Tensor:
    """Reduce (b, c, h, w) → (b, c, 1, 1) by taking max over h, w."""
    raise NotImplementedError()


def _test_ex3():
    x = t.arange(2 * 3 * 4 * 5).reshape(2, 3, 4, 5).float()
    y = ex3_spatial_max(x)
    assert y.shape == (2, 3, 1, 1), f'expected (2,3,1,1), got {y.shape}'
    assert t.allclose(y, x.amax(dim=(2, 3), keepdim=True)), 'values differ from amax(keepdim=True)'
    # Should broadcast — (x - y) must produce no shape error and have x's shape.
    diff = x - y
    assert diff.shape == x.shape, f'broadcast failed: diff shape {diff.shape}'
    _dd_passed.add('ex3')
    print("ex3 ✓")

_test_ex3()

<details><summary>Solution</summary>

```python
def ex3_spatial_max(x: Tensor) -> Tensor:
    return reduce(x, 'b c h w -> b c () ()', 'max')
```

**Why `()`?** Without it the pattern would be `'b c h w -> b c'` and the result would have shape `(b, c)` — same values, but it would not broadcast back against `(b, c, h, w)` because the trailing axes are missing. `()` is the einops equivalent of `keepdim=True`.
</details>

### Exercise 4 — 2×2 average pool (axis decomposition + reduce)

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Bloom level: Apply
> LO: Apply axis decomposition `(h h2)` on the input side combined with reduction over the inner factor — the canonical average-pool pattern.
> Keywords: pooling, decomposition, kwarg-binding
> ```

**KCs targeted:** `reduce-with-decomposition`

Implement `ex4_avg_pool_2x2(x)` to 2×2-average-pool a BCHW tensor.

Input shape: `(b, c, H, W)` where `H` and `W` are even. Output shape: `(b, c, H/2, W/2)`.

Decompose `H` into `(h h2)` and `W` into `(w w2)` on the **left** side. Pass `h2=2, w2=2` as kwargs. On the **right** side keep only `h` and `w` — the `h2` and `w2` axes get reduced over.

Equivalent to `torch.nn.functional.avg_pool2d(x, kernel_size=2, stride=2)`.

In [ ]:
def ex4_avg_pool_2x2(x: Tensor) -> Tensor:
    """2×2 average pool. (b, c, H, W) → (b, c, H/2, W/2)."""
    raise NotImplementedError()


def _test_ex4():
    x = t.arange(2 * 3 * 4 * 4).reshape(2, 3, 4, 4).float()
    y = ex4_avg_pool_2x2(x)
    assert y.shape == (2, 3, 2, 2), f'expected (2,3,2,2), got {y.shape}'
    expected = F.avg_pool2d(x, kernel_size=2, stride=2)
    assert t.allclose(y, expected), 'values differ from F.avg_pool2d(kernel=2, stride=2)'
    _dd_passed.add('ex4')
    print("ex4 ✓")

_test_ex4()

<details><summary>Solution</summary>

```python
def ex4_avg_pool_2x2(x: Tensor) -> Tensor:
    return reduce(x, 'b c (h h2) (w w2) -> b c h w', 'mean', h2=2, w2=2)
```

**Why pass `h2=` and `w2=`?** When you decompose with `(h h2)`, einops needs to know one of the two sizes — the other is inferred from `H`. Naming the inner factor `h2` and binding it via kwarg fixes the pool window size.
</details>

### Exercise 5 — row-wise softmax stabilization (reduce + keepdim + broadcast)

> ```yaml
> Difficulty: 🔴🔴🔴🔴⚪
> Bloom level: Create
> LO: Synthesize reduce + keepdim + broadcast subtraction to perform numerically-stable softmax preprocessing.
> Keywords: softmax-stabilize, broadcast-subtract, integration, multi-kc
> ```

**KCs targeted:** `reduce-pick-aggregator`, `reduce-keepdim-broadcast`, `reduce-normalize-pattern`

Implement `ex5_softmax_stabilize(x)` to subtract the per-row maximum from every element. This is the standard pre-softmax stabilization step that keeps `exp(x)` from overflowing.

Input shape: `(b, n)`. Output shape: `(b, n)`. After your transform, every row's maximum should be exactly `0`.

Use `einops.reduce` with `()` on the row axis so the per-row max keeps a size-1 placeholder, then broadcast-subtract.

> ⚠️ **Integrative exercise.** This combines 3+ KCs in one expression; empirical work (Lohr et al. ITiCSE 2025) shows 3-concept LLM-generated exercises drop from ~94% to ~40% solvability. Expect a step in difficulty here vs Exercises 1-4.

In [ ]:
def ex5_softmax_stabilize(x: Tensor) -> Tensor:
    """Subtract per-row max from x. (b, n) → (b, n).

    After this transform, every row of the result has max == 0.
    """
    raise NotImplementedError()


def _test_ex5():
    x = t.tensor([[1.0, 3.0, 2.0, 5.0], [10.0, 7.0, 8.0, 9.0], [-1.0, -3.0, 0.0, -2.0]])
    y = ex5_softmax_stabilize(x)
    assert y.shape == x.shape, f'shape mismatch: {y.shape} vs {x.shape}'
    row_max = y.amax(dim=1)
    assert t.allclose(row_max, t.zeros(3)), f'row maxes should all be 0, got {row_max}'
    # Softmax-invariant: softmax(x) == softmax(x - row_max)
    assert t.allclose(x.softmax(dim=1), y.softmax(dim=1)), 'softmax-invariance broken'
    _dd_passed.add('ex5')
    print("ex5 ✓")

_test_ex5()

<details><summary>Solution</summary>

```python
def ex5_softmax_stabilize(x: Tensor) -> Tensor:
    row_max = reduce(x, 'b n -> b ()', 'max')
    return x - row_max
```

**Reading the pattern.**
- `'b n -> b ()'` reduces over `n` and keeps a size-1 placeholder, giving `row_max` shape `(b, 1)`.
- `x - row_max` then broadcasts the per-row max across all columns.

**Why this matters.** `exp(x_i)` for large positive `x_i` overflows. Subtracting the per-row max is mathematically a no-op for softmax (numerator and denominator both scale by `exp(-row_max)`) but keeps every exponent ≤ 0, so `exp` stays in `(0, 1]`.
</details>

## Done

Run the cell below to report your progress to Delta Drills. The beacon fires only if all 5 exercises passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex1', 'ex2', 'ex3', 'ex4', 'ex5'}

def _dd_feedback_level(num_passed: int) -> str:
    """Map exercise-pass count → arena-rating feedback enum."""
    if num_passed == 5: return 'not_much'   # 5/5 → felt easy
    if num_passed >= 3: return 'somewhat'
    return 'a_lot'

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {len(missing)} exercises still failing: {sorted(missing)}.")
        print("[Delta Drills] not reporting until all 5 pass.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}',
        'subtopics': [DD_SUBTOPIC],
        'feedback': _dd_feedback_level(len(_dd_passed)),
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()